# Tumor Occurrence Maps (TOMs) — PSMA PET/CT
### Population-Based Mapping of Metastatic Distribution
**EANM 2026 · Yousefirizi & Uribe**

---
**Workflow**

| Step | Description |
|------|-------------|
| 1 | Install & configure |
| 2 | Load cohort CSV |
| 3 | Preprocess (body masking, downsampling) |
| 4 | Select representative template CT |
| 5 | Affine ± B-spline registration of every CT → template |
| 6 | Accumulate lesion masks → probability map |
| 7 | Group TOMs & difference maps |
| 8 | Save outputs + MIP visualisations |

**Inputs** — CSV with columns `case_id, ct_path, mask_path [, group]`  
**Outputs** — `TOM_probability_map.nii.gz`, per-group TOMs, MIPs, QC overlays, `coverage_summary.csv`


In [ ]:
# ─── Cell 1 · Install dependencies ───────────────────────────────────────────
# Run once; restart runtime afterward if needed.
%pip install -q SimpleITK nibabel numpy pandas scipy matplotlib pillow tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 15.7 MB/s eta 0:00:00


In [ ]:
# ─── Cell 2 · Imports & helper functions ─────────────────────────────────────
from __future__ import annotations

import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ── I/O ──────────────────────────────────────────────────────────────────────

def read_image(path: Path) -> sitk.Image:
    """Read NIfTI or DICOM-series from *path* (file or directory)."""
    if path.is_dir():
        reader = sitk.ImageSeriesReader()
        ids = reader.GetGDCMSeriesIDs(str(path))
        if not ids:
            raise RuntimeError(f"No DICOM series found in {path}")
        reader.SetFileNames(reader.GetGDCMSeriesFileNames(str(path), ids[0]))
        return reader.Execute()
    return sitk.ReadImage(str(path))


def write_nifti(img: sitk.Image, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(img, str(path))


# ── Preprocessing ────────────────────────────────────────────────────────────

def body_mask(ct: sitk.Image, hu_thresh: int = -300, morph_r: int = 2) -> sitk.Image:
    """
    Binary body mask from CT via HU threshold → largest connected component
    → morphological closing/opening.
    """
    binary = sitk.Greater(sitk.Cast(ct, sitk.sitkFloat32), float(hu_thresh))
    binary = sitk.BinaryFillhole(binary, foregroundValue=1)
    cc     = sitk.ConnectedComponent(binary)
    stats  = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(cc)
    if stats.GetNumberOfLabels():
        largest = max(stats.GetLabels(), key=stats.GetPhysicalSize)
        binary  = sitk.BinaryThreshold(cc, lowerThreshold=largest,
                                        upperThreshold=largest,
                                        insideValue=1, outsideValue=0)
    mask = sitk.Cast(binary, sitk.sitkUInt8)
    if morph_r > 0:
        mask = sitk.BinaryMorphologicalClosing(mask, [morph_r] * 3)
        mask = sitk.BinaryMorphologicalOpening(mask, [morph_r] * 3)
    return mask


def resample_to_mask_geometry(mask: sitk.Image, ct: sitk.Image) -> sitk.Image:
    """Ensure *mask* shares geometry with *ct* (nearest-neighbour)."""
    return sitk.Resample(mask, ct, sitk.Transform(),
                         sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)


def downsample(img: sitk.Image, factor: int) -> sitk.Image:
    """Integer downsampling; physical extent is preserved."""
    return sitk.Shrink(img, [factor] * img.GetDimension())


def normalise_for_reg(ct: sitk.Image,
                      hu_low: float = -1024, hu_high: float = 1500) -> sitk.Image:
    """Clamp HU + z-score normalisation for mutual-information registration."""
    ct32 = sitk.Cast(sitk.Clamp(ct, lowerBound=hu_low, upperBound=hu_high),
                     sitk.sitkFloat32)
    stats = sitk.StatisticsImageFilter()
    stats.Execute(ct32)
    sigma = stats.GetSigma()
    return (ct32 - stats.GetMean()) / sigma if sigma > 1e-8 else ct32


# ── Registration ─────────────────────────────────────────────────────────────

def register_affine(moving: sitk.Image, fixed: sitk.Image,
                    bins: int = 32) -> sitk.Transform:
    """
    Centroid-initialised, multi-resolution affine registration using
    Mattes mutual information + regular-step gradient descent.
    """
    init = sitk.CenteredTransformInitializer(
        fixed, moving, sitk.AffineTransform(3),
        sitk.CenteredTransformInitializerFilter.GEOMETRY)

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(bins)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsRegularStepGradientDescent(
        learningRate=2.0, minStep=1e-4,
        numberOfIterations=200, relaxationFactor=0.5)
    reg.SetShrinkFactorsPerLevel([4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([2, 1, 0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    reg.SetInitialTransform(init, inPlace=False)
    return reg.Execute(fixed, moving)


def register_bspline(moving_reg: sitk.Image, fixed: sitk.Image,
                     affine_tx: sitk.Transform,
                     grid_spacing_mm: float = 50.0) -> sitk.Transform:
    """
    B-spline (free-form deformable) registration on top of an affine
    initialisation.  *moving_reg* should already be affine-resampled into
    fixed space to allow a clean composite transform.
    """
    phys_size = [sz * sp for sz, sp in zip(fixed.GetSize(), fixed.GetSpacing())]
    mesh_size = [max(1, int(round(s / grid_spacing_mm)) - 1) for s in phys_size]
    bspline   = sitk.BSplineTransformInitializer(
        image1=fixed, transformDomainMeshSize=mesh_size, order=3)

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(32)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsLBFGSB(gradientConvergenceTolerance=1e-5,
                              numberOfIterations=100,
                              maximumNumberOfCorrections=5,
                              maximumNumberOfFunctionEvaluations=1000,
                              costFunctionConvergenceFactor=1e7)
    reg.SetShrinkFactorsPerLevel([2, 1])
    reg.SetSmoothingSigmasPerLevel([1, 0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    reg.SetInitialTransform(bspline, inPlace=False)
    reg.SetMovingInitialTransform(affine_tx)
    reg.SetMetricMovingMask(body_mask(moving_reg))
    reg.SetMetricFixedMask(body_mask(fixed))
    return reg.Execute(fixed, moving_reg)


def apply_transform(img: sitk.Image, ref: sitk.Image,
                    tx: sitk.Transform, interp: int,
                    default_val: float = 0.0,
                    out_type: Optional[int] = None) -> sitk.Image:
    """Resample *img* into *ref* space using *tx*."""
    out = sitk.Resample(img, ref, tx, interp, default_val, img.GetPixelID())
    return sitk.Cast(out, out_type) if out_type is not None else out


# ── Metrics & Visualisation ───────────────────────────────────────────────────

def body_coverage(prob_img: sitk.Image,
                  body_img: sitk.Image, thresh: float = 0.0) -> float:
    """Fraction of body-mask voxels with probability > *thresh* (%)."""
    P = sitk.GetArrayFromImage(prob_img)
    B = sitk.GetArrayFromImage(body_img) > 0
    return float((P[B] > thresh).sum()) / max(B.sum(), 1) * 100.0


def save_mips(tom: sitk.Image, out_dir: Path, tag: str = "TOM") -> None:
    """Save axial / coronal / sagittal maximum-intensity projections as PNG."""
    arr = sitk.GetArrayFromImage(tom).astype(np.float32)
    arr_n = arr / (arr.max() + 1e-8)
    views = {
        "axial":    arr_n.max(axis=0),
        "coronal":  arr_n.max(axis=1),
        "sagittal": arr_n.max(axis=2),
    }
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, proj in views.items():
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(proj, cmap="hot", vmin=0, vmax=1, origin="lower")
        ax.axis("off")
        ax.set_title(f"{tag} — {name} MIP", fontsize=9)
        fig.tight_layout(pad=0.2)
        fig.savefig(out_dir / f"{tag}_{name}.png", dpi=200, bbox_inches="tight")
        plt.close(fig)


def save_qc_overlay(template_ct: sitk.Image, mask_reg: sitk.Image,
                    out_path: Path, case_id: str) -> None:
    """Red-on-grey overlay at the mid-axial slice for visual QC."""
    ct_arr  = sitk.GetArrayFromImage(sitk.Cast(template_ct, sitk.sitkFloat32))
    mk_arr  = sitk.GetArrayFromImage(mask_reg)
    z = ct_arr.shape[0] // 2
    ct_sl = ct_arr[z]
    ct_n  = (ct_sl - ct_sl.min()) / (ct_sl.max() - ct_sl.min() + 1e-8)
    rgb   = np.stack([ct_n, ct_n, ct_n], axis=-1)
    rgb[mk_arr[z] > 0.5, 0] = 1.0          # red channel = tumour
    rgb[mk_arr[z] > 0.5, 1:] *= 0.3

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(rgb, origin="lower")
    ax.set_title(case_id, fontsize=8)
    ax.axis("off")
    fig.tight_layout(pad=0.1)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


print("✓  Helpers loaded.")


✓  Helpers loaded.


## 📁 Mount Drive & auto-build `cases.csv` from your FYR folder

Your data lives at `My Drive > FYR > Images/` (CTs) and `FYR > Labels/` (masks).  
This cell scans those folders, matches CT ↔ label by case ID, and writes `cases.csv`
so Notebook 1 can run without any manual editing.

**autoPET IV PSMA naming convention**
```
Images/psma_<hash>_<date>_0000.nii.gz   ← CT  (channel 0000)
Images/psma_<hash>_<date>_0001.nii.gz   ← PET (channel 0001, not used here)
Labels/psma_<hash>_<date>.nii.gz        ← segmentation mask
```


In [ ]:
# ─── Cell A · Mount Drive & auto-build cases.csv ─────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, re
from pathlib import Path
import pandas as pd

# ── Base paths — adjust only if your folder is named differently ──────────
BASE_DIR    = Path("/content/drive/MyDrive/FYR")
IMAGES_DIR  = BASE_DIR / "Images"
LABELS_DIR  = BASE_DIR / "Labels"
CASES_CSV   = BASE_DIR / "cases_psma.csv"

# Optional: restrict to a subgroup (e.g. "psma") or set to "" for all
MODALITY_PREFIX = "psma"

# Optional: tag all cases with a group label (leave "" for no group column)
GROUP_TAG = ""

# ─────────────────────────────────────────────────────────────────────────────
assert IMAGES_DIR.exists(), f"Images folder not found: {IMAGES_DIR}"
assert LABELS_DIR.exists(), f"Labels folder not found: {LABELS_DIR}"

# ── Index label files by case_id ──────────────────────────────────────────
label_map = {}
for lf in LABELS_DIR.glob("*.nii*"):
    # strip .nii.gz or .nii
    stem = lf.name.replace(".nii.gz", "").replace(".nii", "")
    if MODALITY_PREFIX and not stem.startswith(MODALITY_PREFIX):
        continue
    label_map[stem] = lf          # case_id → label path

print(f"Found {len(label_map)} label files in {LABELS_DIR}")

# ── Match CT files (channel _0000) to labels ─────────────────────────────
rows = []
warn = []
for cf in sorted(IMAGES_DIR.glob("*.nii*")):
    stem = cf.name.replace(".nii.gz", "").replace(".nii", "")
    if MODALITY_PREFIX and not stem.startswith(MODALITY_PREFIX):
        continue

    # autoPET IV: CT is channel _0000, PET is _0001 — skip PET
    if stem.endswith("_0001"):
        continue

    # Derive case_id: strip trailing _0000 if present
    case_id = re.sub(r"_0+0+$", "", stem)   # psma_<hash>_<date>

    if case_id not in label_map:
        warn.append(f"  [WARN] No label for CT: {cf.name}")
        continue

    row = {"case_id": case_id, "ct_path": str(cf), "mask_path": str(label_map[case_id])}
    if GROUP_TAG:
        row["group"] = GROUP_TAG
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(CASES_CSV, index=False)

print(f"\n✓  {len(df)} matched cases  |  {len(warn)} unmatched CTs")
if warn:
    print("\n".join(warn[:10]))
    if len(warn) > 10:
        print(f"  … and {len(warn)-10} more")

print(f"\n✓  CSV written → {CASES_CSV}")
print(f"\nFirst 5 rows:")
display(df.head())

# ── Update CASES_CSV config variable for the rest of the notebook ─────────
CASES_CSV_PATH = str(CASES_CSV)
OUTPUT_DIR_PATH = str(BASE_DIR / "results_psma_TOM")


Mounted at /content/drive
Found 600 label files in /content/drive/MyDrive/FYR/Labels

✓  472 matched cases  |  0 unmatched CTs

✓  CSV written → /content/drive/MyDrive/FYR/cases_psma.csv

First 5 rows:


,case_id,ct_path,mask_path
0,psma_0179419e313f7d8c_2019-06-10,/content/drive/MyDrive/FYR/Images/psma_0179419...,/content/drive/MyDrive/FYR/Labels/psma_0179419...
1,psma_0198cdca94fbb95f_2019-12-28,/content/drive/MyDrive/FYR/Images/psma_0198cdc...,/content/drive/MyDrive/FYR/Labels/psma_0198cdc...
2,psma_0198cdca94fbb95f_2020-05-09,/content/drive/MyDrive/FYR/Images/psma_0198cdc...,/content/drive/MyDrive/FYR/Labels/psma_0198cdc...
3,psma_01a52e26ce5b5e26_2016-11-21,/content/drive/MyDrive/FYR/Images/psma_01a52e2...,/content/drive/MyDrive/FYR/Labels/psma_01a52e2...
4,psma_01a52e26ce5b5e26_2017-04-21,/content/drive/MyDrive/FYR/Images/psma_01a52e2...,/content/drive/MyDrive/FYR/Labels/psma_01a52e2...


## ⚙️ Configuration — edit paths and flags here

In [ ]:
# ─── Cell 3 · Configuration ───────────────────────────────────────────────────
# Paths are set automatically by the auto-discovery cell above.
# You can override them here if needed.

try:
    CASES_CSV  = CASES_CSV_PATH    # set by auto-discovery cell
    OUTPUT_DIR = OUTPUT_DIR_PATH
except NameError:
    # Fallback: set manually if you skipped auto-discovery
    CASES_CSV  = "/content/drive/MyDrive/FYR/cases_psma.csv"
    OUTPUT_DIR = "/content/drive/MyDrive/FYR/results_psma_TOM"

# ── Registration ───────────────────────────────────────────────────────────
DEFORMABLE      : bool  = False   # True → affine + B-spline (slower, more precise)
SHRINK_FACTOR   : int   = 4       # downsampling factor for registration speed
BSPLINE_GRID_MM : float = 50.0    # B-spline control-point spacing (mm)
AFFINE_BINS     : int   = 32      # Mattes MI histogram bins

# ── Preprocessing ─────────────────────────────────────────────────────────
HU_THRESH : int = -300            # body-mask threshold (HU)
MORPH_R   : int = 2               # morphological radius (voxels)

# ── Pipeline behaviour ────────────────────────────────────────────────────
QC_SAMPLES  : int  = 20           # number of QC overlays to save
THREADS     : int  = 8            # CPU threads for SimpleITK

# ── RTSTRUCT (not needed for autoPET IV) ─────────────────────────────────
USE_RTSTRUCT         : bool          = False
RTSTRUCT_NAME_SUBSTR : Optional[str] = None

# ─────────────────────────────────────────────────────────────────────────
from pathlib import Path
from typing import Optional
import SimpleITK as sitk

sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(THREADS)

out = Path(OUTPUT_DIR)
dirs = {k: out / k for k in ("qc", "mips", "groups", "diffs", "transforms")}
for d in dirs.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"Cases CSV  : {CASES_CSV}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Deformable : {DEFORMABLE}  |  Shrink: {SHRINK_FACTOR}×  |  Threads: {THREADS}")


Cases CSV  : /content/drive/MyDrive/FYR/cases_psma.csv
Output dir : /content/drive/MyDrive/FYR/results_psma_TOM
Deformable : False  |  Shrink: 4×  |  Threads: 8


In [ ]:
# ─── Cell 4 · (Optional) Mount Google Drive ───────────────────────────────────
MOUNT_DRIVE = True   # set False if running locally

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ModuleNotFoundError:
        print("Not running in Colab — skipping Drive mount.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ─── Cell 5 · (Optional) RTSTRUCT → NIfTI mask ────────────────────────────────
# Only needed when USE_RTSTRUCT=True.  Install extras with:
#   %pip install -q pydicom rt-utils

if USE_RTSTRUCT:
    try:
        import pydicom
        from rt_utils import RTStructBuilder
        HAS_RT = True
    except ImportError:
        raise ImportError("Install pydicom and rt-utils: %pip install pydicom rt-utils")
else:
    HAS_RT = False


def rtstruct_to_mask(rtstruct_path: Path, dicom_ct_dir: Path,
                     name_substr: Optional[str] = None) -> sitk.Image:
    """Convert the first matching ROI in an RTSTRUCT to a binary NIfTI mask."""
    if not HAS_RT:
        raise RuntimeError("RTSTRUCT support not enabled.")
    rs     = RTStructBuilder.create_from(dicom_series_path=str(dicom_ct_dir),
                                         rt_struct_path=str(rtstruct_path))
    names  = rs.get_roi_names()
    target = next((n for n in names
                   if name_substr and name_substr.lower() in n.lower()), names[0])
    arr    = rs.get_roi_mask_by_name(target).astype(np.uint8)
    ct     = read_image(dicom_ct_dir)
    mask   = sitk.GetImageFromArray(np.transpose(arr, (2, 1, 0)))
    mask.CopyInformation(ct)
    return mask


print("✓  RTSTRUCT helper ready.")


✓  RTSTRUCT helper ready.


## 📂 Load & preprocess cohort

In [ ]:
# ─── Cell 6 · Load cohort CSV and preprocess all cases ───────────────────────
df = pd.read_csv(CASES_CSV)
required = {"case_id", "ct_path", "mask_path"}
if not required.issubset(df.columns):
    raise ValueError(f"CSV must contain columns: {required}")
HAS_GROUP = "group" in df.columns

cases: List[Tuple[str, sitk.Image, sitk.Image, sitk.Image, Optional[str]]] = []
# each entry: (case_id, ct_ds, mask_ds, ct_full_norm, group)

print(f"Loading {len(df)} cases  [shrink={SHRINK_FACTOR}×] …")
for _, row in tqdm(df.iterrows(), total=len(df), desc="Load"):
    cid       = str(row["case_id"])
    ct_full   = read_image(Path(row["ct_path"]))

    if USE_RTSTRUCT:
        mask_full = rtstruct_to_mask(
            Path(row["mask_path"]), Path(row["ct_path"]), RTSTRUCT_NAME_SUBSTR)
    else:
        mask_full = read_image(Path(row["mask_path"]))

    # Harmonise mask geometry to CT before downsampling
    mask_full = resample_to_mask_geometry(mask_full, ct_full)

    ct_ds   = downsample(normalise_for_reg(ct_full), SHRINK_FACTOR)
    mask_ds = downsample(mask_full, SHRINK_FACTOR)

    group = str(row["group"]) if HAS_GROUP else None
    cases.append((cid, ct_ds, mask_ds, group))

print(f"✓  {len(cases)} cases loaded.")


Loading 472 cases  [shrink=4×] …


Load:   0%|          | 0/472 [00:00<?, ?it/s]

✓  472 cases loaded.


## 🗺️ Template selection

The template is the reference anatomical space. We select the scan whose median NCC with all others is highest — a fast proxy for "most representative".

In [ ]:
# ─── Cell 7 · Select representative template CT ───────────────────────────────

def _quick_ncc(a: sitk.Image, b: sitk.Image) -> float:
    """Approximate NCC after resampling *b* into *a* space (identity transform)."""
    b_r  = sitk.Resample(b, a, sitk.Transform(),
                          sitk.sitkLinear, -3.0, b.GetPixelID())
    aa   = sitk.GetArrayFromImage(a).ravel().astype(np.float64)
    bb   = sitk.GetArrayFromImage(b_r).ravel().astype(np.float64)
    denom = aa.std() * bb.std()
    return float(np.corrcoef(aa, bb)[0, 1]) if denom > 1e-8 else 0.0


ct_list = [c[1] for c in cases]

if len(ct_list) <= 50:
    # Full pairwise NCC — feasible for small cohorts
    scores = np.zeros(len(ct_list))
    for i, ci in enumerate(tqdm(ct_list, desc="Template selection")):
        nccs = [_quick_ncc(ci, cj) for j, cj in enumerate(ct_list) if j != i]
        scores[i] = float(np.median(nccs))
    template_idx = int(np.argmax(scores))
else:
    # Shortcut: pick scan closest to volumetric median
    vols  = np.array([float(np.prod(c.GetSize())) for c in ct_list])
    template_idx = int(np.argmin(np.abs(vols - np.median(vols))))

TEMPLATE_CT   = ct_list[template_idx]
template_body = body_mask(TEMPLATE_CT, hu_thresh=HU_THRESH, morph_r=MORPH_R)

write_nifti(template_body,      out / "TOM_body_mask.nii.gz")
write_nifti(TEMPLATE_CT,        out / "TOM_template_ct.nii.gz")

print(f"✓  Template: case #{template_idx}  ({cases[template_idx][0]})")
print(f"   Size    : {TEMPLATE_CT.GetSize()}")
print(f"   Spacing : {[round(s,3) for s in TEMPLATE_CT.GetSpacing()]} mm")


✓  Template: case #5  (psma_024e1c55ab49d220_2014-10-04)
   Size    : (64, 64, 65)
   Spacing : [10.938, 10.938, 13.08] mm


## 🔗 Register all cases → template & accumulate lesion masks

In [ ]:
# ─── Cell 8 · Registration + lesion accumulation ─────────────────────────────

size     = TEMPLATE_CT.GetSize()
acc_all  = np.zeros(sitk.GetArrayFromImage(TEMPLATE_CT).shape, dtype=np.float64)
group_acc: Dict[str, np.ndarray]  = {}
group_cnt: Dict[str, int]         = {}
failed:    List[str]              = []

for i, (cid, ct_ds, mask_ds, group) in enumerate(
        tqdm(cases, desc="Register & accumulate")):
    try:
        # ── 1. Affine ────────────────────────────────────────────────────
        tx_aff = register_affine(ct_ds, TEMPLATE_CT, bins=AFFINE_BINS)

        if DEFORMABLE:
            # ── 2. B-spline on top of affine ─────────────────────────────
            ct_aff = apply_transform(ct_ds, TEMPLATE_CT, tx_aff,
                                     sitk.sitkLinear, default_val=-3.0)
            tx_bsp = register_bspline(ct_aff, TEMPLATE_CT, tx_aff,
                                      grid_spacing_mm=BSPLINE_GRID_MM)
            # Composite: first affine, then bspline correction
            tx_final = sitk.CompositeTransform(3)
            tx_final.AddTransform(tx_aff)
            tx_final.AddTransform(tx_bsp)
        else:
            tx_final = tx_aff

        # ── 3. Warp lesion mask ──────────────────────────────────────────
        mask_reg = apply_transform(
            mask_ds, TEMPLATE_CT, tx_final,
            sitk.sitkNearestNeighbor, default_val=0.0,
            out_type=sitk.sitkUInt8)

        arr = sitk.GetArrayFromImage(mask_reg).astype(np.float64)
        acc_all += arr

        # ── 4. Group accumulation ─────────────────────────────────────────
        if HAS_GROUP and group:
            group_acc.setdefault(group, np.zeros_like(acc_all))
            group_cnt[group] = group_cnt.get(group, 0) + 1
            group_acc[group] += arr

        # ── 5. QC overlay ─────────────────────────────────────────────────
        if i < QC_SAMPLES:
            save_qc_overlay(TEMPLATE_CT, mask_reg,
                            dirs["qc"] / f"{i:03d}_{cid}_zmid.png", cid)

    except Exception as exc:
        warnings.warn(f"[SKIP] {cid}: {exc}")
        failed.append(cid)

N = len(cases) - len(failed)
print(f"\n✓  Accumulated {N} / {len(cases)} cases  "
      f"({'%.1f' % (100*N/len(cases))}% success rate)")
if failed:
    print(f"   Failed ({len(failed)}): {failed[:10]}{'…' if len(failed)>10 else ''}")


Register & accumulate:   0%|          | 0/472 [00:00<?, ?it/s]


✓  Accumulated 472 / 472 cases  (100.0% success rate)


## 🧮 Build probability map & group TOMs

In [ ]:
# ─── Cell 9 · Build TOM probability maps ─────────────────────────────────────

def _make_tom(acc: np.ndarray, n: int,
              ref: sitk.Image) -> sitk.Image:
    """Normalise accumulator → [0,1] probability map."""
    prob = (acc / max(n, 1)).astype(np.float32)
    img  = sitk.GetImageFromArray(prob)
    img.CopyInformation(ref)
    return img


# ── Overall TOM ────────────────────────────────────────────────────────────
tom_all = _make_tom(acc_all, N, TEMPLATE_CT)
write_nifti(tom_all, out / "TOM_probability_map.nii.gz")
save_mips(tom_all, dirs["mips"], tag="TOM_all")

cov_all = body_coverage(tom_all, template_body)
rows = [{"group": "ALL", "n": N, "coverage_pct_P>0": round(cov_all, 3)}]
print(f"Overall TOM  — coverage P>0 : {cov_all:.2f}%")

# ── Per-group TOMs ─────────────────────────────────────────────────────────
group_toms: Dict[str, sitk.Image] = {}
if HAS_GROUP and group_acc:
    print("\nGroup TOMs:")
    for g, acc in group_acc.items():
        ng       = group_cnt[g]
        tom_g    = _make_tom(acc, ng, TEMPLATE_CT)
        group_toms[g] = tom_g
        write_nifti(tom_g, dirs["groups"] / f"{g}_TOM.nii.gz")
        save_mips(tom_g, dirs["mips"], tag=f"TOM_{g}")
        cov_g = body_coverage(tom_g, template_body)
        rows.append({"group": g, "n": ng, "coverage_pct_P>0": round(cov_g, 3)})
        print(f"  {g:>20s}  n={ng:>4d}  coverage={cov_g:.2f}%")

    # ── Pairwise difference maps (all pairs, or just the two-group case) ─
    group_names = list(group_acc.keys())
    pairs = [(group_names[i], group_names[j])
             for i in range(len(group_names))
             for j in range(i+1, len(group_names))]
    for g1, g2 in pairs:
        diff_arr = (sitk.GetArrayFromImage(group_toms[g1]) -
                    sitk.GetArrayFromImage(group_toms[g2]))
        diff_img = sitk.GetImageFromArray(diff_arr)
        diff_img.CopyInformation(TEMPLATE_CT)
        tag = f"{g1}_vs_{g2}"
        write_nifti(diff_img, dirs["diffs"] / f"{tag}_diff.nii.gz")
        save_mips(diff_img, dirs["mips"], tag=f"DIFF_{tag}")

# ── Coverage summary ───────────────────────────────────────────────────────
summary = pd.DataFrame(rows)
summary.to_csv(out / "coverage_summary.csv", index=False)
print("\nCoverage summary:")
display(summary)
print(f"\n✓  All outputs saved → {out}")


Overall TOM  — coverage P>0 : 20.52%

Coverage summary:


,group,n,coverage_pct_P>0
0,ALL,472,20.524



✓  All outputs saved → /content/drive/MyDrive/FYR/results_psma_TOM
